In [1]:
import sys

sys.path.append("..")

import os

from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [2]:
from groq import Groq

groq_client = Groq(api_key=GROQ_API_KEY)
print("Groq client ready")

Groq client ready


In [3]:
audio_path = "../models/test/sample_query.mp3"

with open(audio_path, "rb") as audio_file:
    transcription = groq_client.audio.transcriptions.create(
        file=audio_file,
        model="whisper-large-v3-turbo",
    )

transcribed_text = transcription.text.strip()
print(f"Transcribed: {transcribed_text!r}")

Transcribed: 'Is wireless earbuds in stock?'


In [6]:
from app.core.config import get_settings
from app.core.models import load_shared_models
from app.domain.fulfillment.model_registry import load_fulfillment_models
from app.domain.nlp.model_registry import load_nlp_models
from app.domain.rag.model_registry import load_rag_models
from app.domain.supervisor.model_registry import load_supervisor_models
from app.domain.vision.model_registry import load_vision_models

settings = get_settings()
settings.vision_weights_path = "../models/shelf_detection_best.pt"
shared = load_shared_models(settings)
nlp_models = load_nlp_models(settings, shared)
rag_models = load_rag_models(settings, shared)
fulfillment_models = load_fulfillment_models(settings, shared)
vision_models = load_vision_models(settings, shared)

supervisor_models = load_supervisor_models(
    settings, shared, nlp_models, rag_models, fulfillment_models, vision_models
)

result = supervisor_models.graph.invoke({
    "query": transcribed_text,
    "image_path": None,
    "category": None, "confidence": None, "method": None,
    "response": None, "needs_image": False,
})

response_text = result["response"]
print(f"Routed to: {result['category']}")
print(f"Response: {response_text}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Routed to: fulfillment
Response: I’m sorry, but our wireless Bluetooth earbuds are currently out of stock. Let me know if you’d like to be notified when they’re back or if you’d like help finding a similar product!


In [15]:
speech_response = groq_client.audio.speech.create(
    model="canopylabs/orpheus-v1-english",
    voice="autumn", 
    input=response_text,
    response_format="wav",
)

output_path = "../models/test/response_output.wav"
speech_response.write_to_file(output_path)
print(f"Audio response saved to: {output_path}")

Audio response saved to: ../models/test/response_output.wav


In [16]:
from IPython.display import Audio, display

display(Audio(output_path))